[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

In [ ]:
# 在 Colab 上运行时取消注释
#!pip install dm-haiku
#!pip install optax

In [ ]:
import jax
import jax.numpy as jnp
import haiku as hk
from functools import partial
import math
import numpy as np
from numpy.random import random
import optax
import torch
from typing import Any, Sequence

# 玩玩 Jax/Haiku/Optax

在这个 notebook 里，我通过重新实现 [dataflowr 模块 2b](https://dataflowr.github.io/website/modules/2b-automatic-differentiation/) 的线性回归例子，来学习 [JAX](https://jax.readthedocs.io/en/latest/index.html) 以及 [Haiku](https://dm-haiku.readthedocs.io/en/latest/index.html) 和 [Optax](https://optax.readthedocs.io/en/latest/index.html) 这两个库。

我觉得 Sabrina Mielke 的这篇文章非常有用：[From PyTorch to JAX: towards neural net frameworks that purify stateful code](https://sjmielke.com/jax-purify.htm)


In [ ]:
# 生成随机输入数据
x = random((30,2)).astype('float32')
# 根据输入数据 x 生成对应的标签
y = np.dot(x, [2., -3.]) + 1.
y = np.expand_dims(y, axis=1).astype('float32')
w_source = np.array([2., -3.])
b_source  = np.array([1.])

# 准备工作

我们的模型是：
$$
y_t = 2x^1_t-3x^2_t+1, \quad t\in\{1,\dots,30\}
$$

我们的任务是在给定'观测值' $(x_t,y_t)_{t\in\{1,\dots,30\}}$ 的情况下，恢复出权重 $w^1=2, w^2=-3$ 和偏置 $b = 1$。

为此，我们要解下面这个优化问题：
$$
\underset{w^1,w^2,b}{\operatorname{argmin}} \sum_{t=1}^{30} \left(w^1x^1_t+w^2x^2_t+b-y_t\right)^2
$$


In [ ]:
# 随机初始化可学习的权重和偏置
w_init = random(2)
b_init = random(1)

w = w_init
b = b_init
print("initial values of the parameters:", w, b )

dtype = torch.FloatTensor
w_init_t = torch.from_numpy(w_init).type(dtype)
b_init_t = torch.from_numpy(b_init).type(dtype)
x_t = torch.from_numpy(x).type(dtype)
y_t = torch.from_numpy(y).type(dtype)

learning_rate = 1e-2

# PyTorch 版本

PyTorch 实现取自 [Dataflowr 模块 2b](https://dataflowr.github.io/website/modules/2b-automatic-differentiation/)


In [ ]:
model = torch.nn.Sequential(torch.nn.Linear(2, 1),)

for m in model.children():
    m.weight.data = w_init_t.clone().unsqueeze(0)
    m.bias.data = b_init_t.clone()

loss_fn = torch.nn.MSELoss(reduction='sum')

model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(10):
    y_pred = model(x_t)
    loss = loss_fn(y_pred, y_t)
    print("progress:", "epoch:", epoch, "loss",loss.item())
    # 梯度清零、反向传播、更新权重。
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
# 训练结束后
print("estimation of the parameters:")
for param in model.parameters():
    print(param)

# 用 Haiku 做 Jax 线性层

玩一下线性层……你先定义自己的 python 函数，然后借助 `hk.transfrom` 把它'jax 化'成一个纯函数。


In [ ]:
def _linear(x, config):
    return hk.Linear(config.size_out)(x)

class toy_config:
    size_out = 5

linear = hk.without_apply_rng(hk.transform(lambda x: _linear(x, config=toy_config)))

In [ ]:
rng_key = jax.random.PRNGKey(42)
x_dummy = jax.random.normal(key=rng_key, shape=(1,7))

In [ ]:
params = linear.init(rng=rng_key, x=x_dummy)

In [ ]:
params

In [ ]:
x_in = jax.random.normal(key=rng_key, shape=(10,3,7))
out = linear.apply(x=x_in, params= params)

In [ ]:
out.shape

# Haiku：带初始化的线性层

我想和 PyTorch 实现对比验证结果，所以参数需要从相同的初始值开始。


In [ ]:
# 参考 https://github.com/deepmind/dm-haiku/blob/main/haiku/_src/initializers.py#L51#L63
class Init_jnparray(hk.initializers.Initializer):
    def __init__(self, w: jnp.ndarray):
        self.w = w

    def __call__(self, shape: Sequence[int], dtype: Any) -> jnp.ndarray:
        if self.w.shape != tuple(shape):
            raise ValueError('Error in shape! w:', self.w.shape,' and shape:', shape)
        return self.w.astype(dtype)

In [ ]:
class config:
    size_out = 1
    w_source = jnp.array([w_init]).swapaxes(1,0)
    b_source = jnp.array(b_init)


def _linear(x, config):
    return hk.Linear(config.size_out,w_init=Init_jnparray(config.w_source), b_init=Init_jnparray(config.b_source))(x)

注意这里没有任何随机成分（没有随机初始化）。


In [ ]:
x_dummy = jax.random.normal(key=rng_key, shape=(1,2))
linear = hk.without_apply_rng(hk.transform(lambda x: _linear(x, config=config)))
params = linear.init(x=x_dummy,rng=None)

In [ ]:
params

In [ ]:
out = linear.apply(x=x,params=params)

In [ ]:
out.shape

# 计算损失和梯度


In [ ]:
def mse_loss(y_pred, y_t):
    return jax.lax.integer_pow(y_pred - y_t,2).sum()

mse_loss(out,y)

In [ ]:
def loss_fn(x_in, y_t, config):
    return mse_loss(_linear(x=x_in, config=config),y_t)

In [ ]:
hk_loss_fn = hk.without_apply_rng(hk.transform(partial(loss_fn, config=config)))

In [ ]:
params = hk_loss_fn.init(rng=rng_key, x_in=x,y_t=y)

In [ ]:
params

把损失重新定义成变换后的 haiku 损失的 `apply` 方法，然后，PyTorch 里的 `backward` 操作在 jax 中用 [`value_and_grad`](https://jax.readthedocs.io/en/latest/notebooks/autodiff_cookbook.html#evaluate-a-function-and-its-gradient-using-value-and-grad) 完成。


In [ ]:
loss_fn = hk_loss_fn.apply
loss, grads = jax.value_and_grad(loss_fn)(params,x_in=x,y_t=y)

In [ ]:
loss

In [ ]:
grads

# 优化器

现在事情就很好理解了


In [ ]:
optimizer = optax.sgd(learning_rate=1e-2)

In [ ]:
opt_state = optimizer.init(params)
for epoch in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params,x_in=x,y_t=y)
    print("progress:", "epoch:", epoch, "loss",loss)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
# 训练结束后
print("estimation of the parameters:")
print(params)

# Jax/Haiku/Optax 完整代码


In [ ]:
class config:
    size_out = 1
    w_source = jnp.array([w_init]).swapaxes(1,0)
    b_source = jnp.array(b_init)

class Init_jnparray(hk.initializers.Initializer):
    def __init__(self, w: jnp.ndarray):
        self.w = w

    def __call__(self, shape: Sequence[int], dtype: Any) -> jnp.ndarray:
        if self.w.shape != tuple(shape):
            raise ValueError('Error in shape! w:', self.w.shape,' and shape:', shape)
        return self.w.astype(dtype)
    
def _linear(x, config):
    return hk.Linear(config.size_out,w_init=Init_jnparray(config.w_source), b_init=Init_jnparray(config.b_source))(x)

def mse_loss(y_pred, y_t):
    return jax.lax.integer_pow(y_pred - y_t,2).sum()

def loss_fn(x_in, y_t, config):
    return mse_loss(_linear(x=x_in, config=config),y_t)

hk_loss_fn = hk.without_apply_rng(hk.transform(partial(loss_fn, config=config)))
params = hk_loss_fn.init(x_in=x,y_t=y,rng=None)
loss_fn = hk_loss_fn.apply

optimizer = optax.sgd(learning_rate=1e-2)

opt_state = optimizer.init(params)
for epoch in range(10):
    loss, grads = jax.value_and_grad(loss_fn)(params,x_in=x,y_t=y)
    print("progress:", "epoch:", epoch, "loss",loss)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
# 训练结束后
print("estimation of the parameters:")
print(params)

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)